# 경기도 카드 소비 데이터 매출 예측 모델 학습

## 사전 준비
1. 캐글 데이터셋에 `dataset/` 폴더의 CSV 파일들을 업로드
2. 이 노트북에서 해당 데이터셋을 추가
3. `Run All` 실행 후 output에서 모델 파일 다운로드

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 캐글 데이터셋 경로 (업로드한 데이터셋 이름에 맞게 수정)
DATA_DIR   = "/kaggle/input/gyeonggi-card-data"  # 데이터셋 경로
OUTPUT_DIR = "/kaggle/working"

os.makedirs(f"{OUTPUT_DIR}/model",    exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/encoders", exist_ok=True)

print("데이터 파일 목록:")
files = sorted(glob.glob(f"{DATA_DIR}/tbsh_gyeonggi_day_*.csv"))
print(f"  총 {len(files)}개")
for f in files[:5]:
    print(f"  {os.path.basename(f)}")

In [ ]:
# =========================================================
# 설정
# =========================================================
MODEL_FEATURES = ["sex", "age", "day", "hour", "month",
                  "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2", "cnt"]
LABEL_COLS  = ["age", "day", "hour", "month"]
ONEHOT_COLS = ["sex", "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2"]

TOTAL_SAMPLE   = 1_000_000   # 총 샘플 수
REMOVE_OUTLIER = True

In [ ]:
# =========================================================
# 1) 파일별 행 수 파악 → 비율 샘플링
# =========================================================
print("파일별 행 수 파악 중...")
file_sizes = {}
for f in files:
    df_tmp = pd.read_csv(f, encoding="utf-8-sig", usecols=["amt"])
    file_sizes[f] = len(df_tmp)

total_rows = sum(file_sizes.values())
print(f"전체 행 수: {total_rows:,}")

# 파일별 샘플 수 = 전체 샘플 × (파일 행 수 / 전체 행 수)
file_sample_n = {
    f: max(500, int(TOTAL_SAMPLE * size / total_rows))
    for f, size in file_sizes.items()
}
print("\n도시별 샘플 수 (상위 5개):")
for f, n in sorted(file_sample_n.items(), key=lambda x: -x[1])[:5]:
    city = os.path.basename(f).split("_")[-1].replace(".csv", "")
    print(f"  {city}: {n:,}")

In [ ]:
# =========================================================
# 2) 데이터 로드 + 전처리
# =========================================================
dfs = []
for i, f in enumerate(files, 1):
    city = os.path.basename(f).split("_")[-1].replace(".csv", "")
    df = pd.read_csv(f, encoding="utf-8-sig",
                     usecols=["ta_ymd", "admi_cty_no", "card_tpbuz_nm_1",
                               "card_tpbuz_nm_2", "hour", "sex", "age", "day",
                               "amt", "cnt"])
    df["month"] = df["ta_ymd"].astype(str).str[4:6].astype(int)
    df["admi_cty_no"] = df["admi_cty_no"].astype(str)
    df["log_amt"] = np.log1p(df["amt"])

    if REMOVE_OUTLIER:
        upper = df["amt"].quantile(0.99)
        df = df[df["amt"] <= upper]

    df = df[MODEL_FEATURES + ["log_amt"]].dropna()

    n = file_sample_n[f]
    if len(df) > n:
        df = df.sample(n, random_state=42)

    dfs.append(df)
    print(f"[{i}/{len(files)}] {city}: {len(df):,}행")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\n최종 학습 데이터: {len(df_all):,}행")

In [ ]:
# =========================================================
# 3) 인코딩
# =========================================================
X = df_all[MODEL_FEATURES].copy()
y = df_all["log_amt"]

for col in LABEL_COLS:
    X[col] = X[col].astype(str)

label_encoders = {}
for col in LABEL_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_arr   = ohe.fit_transform(X[ONEHOT_COLS].astype(str))
ohe_names = ohe.get_feature_names_out(ONEHOT_COLS).tolist()
ohe_df    = pd.DataFrame(ohe_arr, columns=ohe_names, index=X.index)

# cnt는 수치형 그대로 사용
encoded_X = pd.concat([X[LABEL_COLS + ["cnt"]].reset_index(drop=True),
                       ohe_df.reset_index(drop=True)], axis=1)
feature_columns = encoded_X.columns.tolist()

print(f"피처 수: {len(feature_columns)}개")

In [ ]:
# =========================================================
# 4) LightGBM 학습
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    encoded_X, y, test_size=0.2, random_state=42)

print(f"학습 데이터: {len(X_train):,}행 / 검증 데이터: {len(X_test):,}행")

t0 = time.time()
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(100)
    ]
)
print(f"학습 완료: {time.time()-t0:.1f}초")

In [ ]:
# =========================================================
# 5) 평가
# =========================================================
pred = model.predict(X_test)
y_real    = np.expm1(y_test)
pred_real = np.maximum(np.expm1(pred), 0)

r2   = r2_score(y_real, pred_real)
rmse = np.sqrt(mean_squared_error(y_real, pred_real))
mae  = mean_absolute_error(y_real, pred_real)

print(f"R²   = {r2:.4f}")
print(f"RMSE = {rmse:,.0f}")
print(f"MAE  = {mae:,.0f}")

In [ ]:
# =========================================================
# 6) 저장
# =========================================================
joblib.dump(model,           f"{OUTPUT_DIR}/model/sales_predict_model.pkl")
joblib.dump(label_encoders,  f"{OUTPUT_DIR}/encoders/label_encoders.pkl")
joblib.dump(ohe,             f"{OUTPUT_DIR}/encoders/onehot_encoder.pkl")
joblib.dump(feature_columns, f"{OUTPUT_DIR}/encoders/feature_columns.pkl")
joblib.dump({
    "model_name": "LightGBM",
    "R2": r2, "RMSE": rmse, "MAE": mae,
    "sample_size": TOTAL_SAMPLE,
    "features": MODEL_FEATURES,
    "use_log_target": True,
    "remove_outliers": REMOVE_OUTLIER,
}, f"{OUTPUT_DIR}/model/model_info.pkl")

print("저장 완료!")
print(f"  /kaggle/working/model/sales_predict_model.pkl")
print(f"  /kaggle/working/encoders/ (3개 파일)")
print("\n오른쪽 Output 탭에서 파일 다운로드하세요.")